# Day 4

## Tokenizing with code

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [2]:
tokens

[12194, 922, 1308, 382, 6117, 326, 357, 1299, 9171, 26458, 5148]

In [3]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
6117 =  Ed
326 =  and
357 =  I
1299 =  like
9171 =  ban
26458 = offee
5148 =  pie


In [ ]:
encoding.decode([326])

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [1]:
from openai import OpenAI

openai = OpenAI()

In [12]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [4]:
!ollama ps
!ollama pull llama3:8b

NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 
pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠇ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏  71 KB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏ 4.3 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏  11 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   0% ▕                  ▏  18 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  23 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  36 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a:   1% ▕                  ▏  41 MB/4.7 GB                  pulling manifest 
pulling 6a0746a1ec1a: 

### A message to OpenAI is a list of dicts

In [16]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

In [17]:
response = ollama.chat.completions.create(model="llama3:8b", messages=messages)
response.choices[0].message.content

"Nice to meet you, Ed! I'm here to help with any questions or tasks you might have. What's on your mind today? Do you need some advice, assistance with a project, or just want to chat about something interesting? Let me know and I'll do my best to support you!"

### OK let's now ask a follow-up question

In [14]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

#### Or alternatily locall (ollama)

In [15]:
response = ollama.chat.completions.create(model="llama3:8b", messages=messages)
response.choices[0].message.content

"I'm happy to help! Unfortunately, I don't know your name yet. You can tell me, and I'll make sure to remember it for our conversation!"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [18]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [19]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Your name is Ed! How can I help you today, Ed?'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!



# WRAS-NOTE: OpenAI Chat Message Roles

- **system**: High-level instructions that set the assistant's behaviour, tone, and constraints (usually sent once at conversation start).
- **user**: The human user's messages or prompts — what you want the assistant to do.
- **assistant**: The model's previous responses; include these to continue a conversation or provide context (creates the illusion of memory).
- **function**: Output returned by a tool or external function. Function messages include a `name` (the tool) and `content` (the result).

Function-call flow (brief):
1. The model may emit an assistant message requesting a `function_call` specifying which tool to run and with what arguments.
2. Your code runs the tool and returns its output as a message with `role: "function"` and the `name` of the tool.
3. You send the function message back to the model so it can incorporate the tool result into the next assistant reply.

Tip: Every message is a dict with at least `role` and `content`; function messages also include `name`.

<!-- WRAS-NOTE -->

# WRAS-NOTE: Function-call JSON Example

Below is a short JSON snippet showing the function-call sequence (model requests a function, you run it, return the function output, then the model replies):

```json
{
  "messages": [
    { "role": "system", "content": "You are a helpful assistant." },
    { "role": "user", "content": "What's the weather in Berlin?" },
    { "role": "assistant", "content": null, "function_call": { "name": "get_weather", "arguments": "{\"location\":\"Berlin\"}" } },
    { "role": "function", "name": "get_weather", "content": "{\"temp\":\"5C\",\"condition\":\"Cloudy\"}" },
    { "role": "assistant", "content": "The weather in Berlin is Cloudy and 5°C." }
  ]
}
```

Short flow: model -> (assistant.function_call) -> run tool -> return as `function` message -> model final reply.

<!-- WRAS-NOTE -->